In [1]:
import cv2
import PIL
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from random import sample
from matplotlib.pyplot import figure
from sklearn.model_selection import StratifiedKFold
%matplotlib inline

### Images

In [2]:
def plotRawImages(label):
    size, rows, cols = 64, 5, 5
    data = pd.read_csv('../../data/raw/train.csv')
    data = data[data['label'] == label]
    data = sample(data['image_id'].tolist(), 25)
    path = '../../data/raw/train_images/'
    data = [path + x for x in data]
    mosaic = PIL.Image.new(mode='RGB', size=(size*cols + (cols-1), size*rows + (rows-1)))
    for idx, name in enumerate(data):
        img = cv2.imread(name, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ix  = idx % cols
        iy  = idx // cols
        img = np.clip(img, 0, 255).astype(np.uint8)
        img = PIL.Image.fromarray(img)
        img = img.resize((size, size), resample=PIL.Image.BILINEAR)
        mosaic.paste(img, (ix*size + ix, iy*size + iy))
    plt.figure(figsize=(12,12))
    plt.imshow(mosaic)
    return None

In [3]:
# plotRawImages(0)

In [4]:
# plotRawImages(1)

In [5]:
# plotRawImages(2)

In [6]:
# plotRawImages(3)

### Size

In [25]:
def getSize(image):
    path = '../../data/merged/train_images/'
    image = path + image
    image = cv2.imread(image, cv2.IMREAD_COLOR)
    height, width, _ = image.shape
    return (height, width)

In [26]:
data = pd.read_csv('../../data/merged/merged.csv')
data = data[data['source'] == 2019]
data['shape'] = data['image_id'].map(lambda x : getSize(x))
data['height'] = data['shape'].map(lambda x : x[0])
data['width'] = data['shape'].map(lambda x : x[1])

In [27]:
data.height.value_counts()

500     3252
666      924
888      315
889        8
512        7
        ... 
683        1
534        1
538        1
582        1
1071       1
Name: height, Length: 216, dtype: int64

In [28]:
data.width.value_counts()

500     2104
888      800
625      634
666      492
499       20
        ... 
1071       1
1005       1
941        1
925        1
424        1
Name: width, Length: 350, dtype: int64

### Split

In [29]:
data = pd.read_csv('../../data/merged/merged.csv')
data = data.reset_index(drop=True)

In [30]:
split = StratifiedKFold(n_splits=5, random_state=2017, shuffle=True)

In [31]:
data['fold'] = -1

In [32]:
counter = 0
for train_idx, val_idx in split.split(data['image_id'], data['label']):
    data.loc[val_idx,'fold'] = counter
    counter += 1

In [33]:
data['fold'].value_counts()

1    5268
0    5268
4    5267
3    5267
2    5267
Name: fold, dtype: int64

In [34]:
data.label.value_counts()

3    15462
1     3476
2     3017
4     2890
0     1492
Name: label, dtype: int64

In [35]:
data.label.value_counts() / len(data)

3    0.587083
1    0.131982
2    0.114554
4    0.109732
0    0.056650
Name: label, dtype: float64

In [17]:
data.pivot_table(index='fold', columns='label', values='image_id', aggfunc='count')

label,0,1,2,3,4
fold,,,,,
0,299,695,604,3092,578
1,299,695,604,3092,578
2,298,695,603,3093,578
3,298,695,603,3093,578
4,298,696,603,3092,578


In [18]:
data.to_csv('../../data/merged/data.csv', index=False)

In [19]:
data.head()

,image_id,label,source,fold
0,1000015157.jpg,0,2020,4
1,1000201771.jpg,3,2020,0
2,100042118.jpg,1,2020,2
3,1000723321.jpg,1,2020,4
4,1000812911.jpg,3,2020,2


In [20]:
data.shape

(26337, 4)

In [21]:
data['image_id'].nunique()

26337

In [22]:
data.source.value_counts()

2020    21397
2019     4940
Name: source, dtype: int64